# Measure Qwen latency on a cloud GPU using vLLM

In Google Colab, select **Runtime > Change runtime type > GPU**, then run
these cells in order. GPU access depends on your account and availability.
No cloud account credentials, repository access, or project files are needed.
Installation and model weights are downloaded into this notebook runtime.

The default model is Qwen3 4B Instruct, matching the intended planner family.
Its float16 weights differ from quantized Ollama weights: do not attribute
a latency difference solely to the runtime. Use `Qwen/Qwen3-0.6B` for a
smaller smoke run; that measures a different model.

This notebook measures streaming LLM serving latency with synthetic numbers.
It does not run the application's diagnostic, retrieval, or tool workflow.
The server binds to loopback. No public endpoint or tunnel is created.
Stop the server and disconnect/delete the runtime when finished.

References: [vLLM GPU requirements](https://docs.vllm.ai/en/latest/getting_started/installation/gpu/),
[vLLM server](https://docs.vllm.ai/en/latest/cli/serve/),
[Colab GPU availability](https://research.google.com/colaboratory/faq.html).

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time, urllib.request

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
VLLM_PACKAGE = "vllm"  # Exact installed versions are saved with the results.
REPEATS, WARMUPS, MAX_TOKENS = 20, 2, 128
work = pathlib.Path("/content/vllm-latency")
work.mkdir(parents=True, exist_ok=True)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True, check=True).stdout.strip()
print("GPU:", gpu)
assert sys.version_info >= (3, 11), "Use a current Python 3.11+ runtime"

Installation can take several minutes. vLLM uses its own Python environment
so notebook preinstalled Torch/Transformers packages do not determine its dependencies.
GPU/driver compatibility is checked by the server; inspect `server.log` if it fails.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=True)
env = work / "env"
subprocess.run([sys.executable, "-m", "virtualenv", str(env)], check=True)
python = str(env / "bin" / "python")
subprocess.run([python, "-m", "pip", "install", VLLM_PACKAGE], check=True)
freeze = subprocess.run([python, "-m", "pip", "freeze"], capture_output=True,
                        text=True, check=True).stdout
(work / "requirements-resolved.txt").write_text(freeze)

In [ ]:
# Rerunning this cell stops only the server started by this notebook.
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
envvars = dict(os.environ, VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1",
               HF_HUB_DISABLE_TELEMETRY="1")
help_text = subprocess.run([str(env / "bin" / "vllm"), "serve", "--help"],
                           capture_output=True, text=True, check=True, env=envvars).stdout
# vLLM versions use either spelling for disabling automatic prefix caching.
cache_flag = ("--disable-prefix-caching" if "--disable-prefix-caching" in help_text
              else "--no-enable-prefix-caching")
command = [str(env / "bin" / "vllm"), "serve", MODEL,
           "--host", "127.0.0.1", "--port", "8001", "--dtype", "half",
           "--max-model-len", "8192", "--max-num-seqs", "1",
           "--gpu-memory-utilization", "0.85", "--enforce-eager", cache_flag]
(work / "server-command.json").write_text(json.dumps(command, indent=2))
log = (work / "server.log").open("w")
server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=envvars)
endpoint = "http://127.0.0.1:8001/v1"
startup = time.monotonic()
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
try:
    while time.monotonic() - startup < 1200:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited. See server.log below.")
        try:
            with opener.open(endpoint + "/models", timeout=3) as response:
                available = json.load(response)
            if MODEL in {item["id"] for item in available["data"]}:
                break
        except (OSError, ValueError):
            pass
        print("Waiting for model download/load:", round(time.monotonic() - startup), "seconds", flush=True)
        time.sleep(10)
    else:
        raise TimeoutError("Model startup exceeded 20 minutes")
    print("Ready:", available)
    print("Startup seconds (excluded from serving latency):", round(time.monotonic() - startup, 2))
except Exception:
    if server.poll() is None:
        server.terminate()
    print((work / "server.log").read_text()[-10000:])
    raise

Run two warmups and twenty measured requests at concurrency one. Record
median/p95 time to first visible token and total response time. Decode speed
is an estimate based on server usage counts and client stream timestamps.
`--enforce-eager` and disabled prefix caching are reproducible starter settings,
not a claim of optimal vLLM performance. Missing counts remain unknown;
errors are retained and excluded from successful latency percentiles.

In [ ]:
client_source = '"""Measure streaming OpenAI-compatible inference using synthetic inputs only.\n\nStandard library only; works beside vLLM on a GPU VM or in Colab. This measures\nclient-observed serving latency, not the application\'s full diagnostic workflow.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport ipaddress\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\n\nPROMPT = (\n    "These are synthetic example numbers, not plant measurements. "\n    "For [10, 20, 30], explain how to calculate the mean, multiply it by eight, "\n    "and convert that value from kPa to bar. Show the arithmetic and units."\n)\n\n\nclass NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\n\ndef normalize_endpoint(endpoint, allow_remote=False):\n    parsed = urllib.parse.urlsplit(endpoint)\n    if (parsed.scheme not in {"http", "https"} or not parsed.hostname\n            or parsed.username is not None or parsed.password is not None\n            or parsed.query or parsed.fragment):\n        raise ValueError("Use an HTTP(S) base URL without credentials, query or fragment")\n    try:\n        local = ipaddress.ip_address(parsed.hostname).is_loopback\n    except ValueError:\n        local = parsed.hostname.lower() in {"localhost", "vllm"}\n    if not local and not (allow_remote and parsed.scheme == "https"):\n        raise ValueError("Remote benchmarking requires --allow-remote and an HTTPS endpoint")\n    base = endpoint.rstrip("/")\n    return base if base.endswith("/v1") else base + "/v1"\n\n\ndef percentile(values, percent):\n    """Linearly interpolated sample percentile; small samples remain descriptive."""\n    if not values:\n        return None\n    ordered = sorted(values)\n    index = (len(ordered) - 1) * percent / 100\n    low, high = math.floor(index), math.ceil(index)\n    return round(ordered[low] + (ordered[high] - ordered[low]) * (index - low), 6)\n\n\ndef summarize(samples):\n    good = [sample for sample in samples if sample["success"]]\n    summary = {"attempted": len(samples), "successful": len(good),\n               "failed": len(samples) - len(good)}\n    for key in ("ttft_seconds", "total_seconds", "decode_tokens_per_second_estimate"):\n        values = [sample[key] for sample in good if sample.get(key) is not None]\n        summary[key] = {"count": len(values), "p50": percentile(values, 50),\n                        "p95": percentile(values, 95)}\n    return summary\n\n\ndef stream_sample(opener, endpoint, model, api_key, max_tokens, timeout):\n    payload = {"model": model, "messages": [{"role": "user", "content": PROMPT}],\n               "temperature": 0, "max_tokens": max_tokens, "stream": True,\n               "stream_options": {"include_usage": True},\n               "chat_template_kwargs": {"enable_thinking": False}}\n    request = urllib.request.Request(\n        endpoint + "/chat/completions", data=json.dumps(payload).encode(),\n        headers={"Content-Type": "application/json", "Authorization": f"Bearer {api_key}"})\n    started = time.perf_counter()\n    first = last = None\n    usage, finish_reason = {}, None\n    saw_done = False\n    with opener.open(request, timeout=timeout) as response:\n        for line in response:\n            now = time.perf_counter()\n            if now - started > timeout:\n                raise TimeoutError("Total request deadline exceeded")\n            if not line.startswith(b"data:"):\n                continue\n            raw = line[5:].strip()\n            if raw == b"[DONE]":\n                saw_done = True\n                break\n            if not raw:\n                continue\n            event = json.loads(raw)\n            if event.get("error"):\n                raise ValueError("Server returned an error event")\n            if isinstance(event.get("usage"), dict):\n                usage = event["usage"]\n            choices = event.get("choices") or []\n            if choices:\n                choice = choices[0]\n                finish_reason = choice.get("finish_reason") or finish_reason\n                if (choice.get("delta") or {}).get("content"):\n                    first = now if first is None else first\n                    last = now\n    if not saw_done or first is None:\n        raise ValueError("Stream did not finish with [DONE] and nonempty visible output")\n    count = usage.get("completion_tokens")\n    prompt_count = usage.get("prompt_tokens")\n    count = count if type(count) is int and count > 0 else None\n    prompt_count = prompt_count if type(prompt_count) is int and prompt_count >= 0 else None\n    decode = last - first\n    return {"success": True, "ttft_seconds": round(first - started, 6),\n            "total_seconds": round(time.perf_counter() - started, 6),\n            "prompt_tokens": prompt_count, "completion_tokens": count,\n            "finish_reason": finish_reason,\n            "decode_tokens_per_second_estimate": (\n                round((count - 1) / decode, 3) if count and count > 1 and decode > 0 else None)}\n\n\ndef hardware_info():\n    info = {"client_platform": platform.platform(), "client_python": platform.python_version()}\n    if shutil.which("nvidia-smi"):\n        try:\n            result = subprocess.run(\n                ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],\n                capture_output=True, text=True, timeout=10,\n                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0), check=True)\n            info["client_gpus"] = result.stdout.strip().splitlines()\n        except (OSError, subprocess.SubprocessError):\n            info["client_gpus"] = None\n    return info\n\n\ndef positive_int(value):\n    parsed = int(value)\n    if parsed < 1:\n        raise argparse.ArgumentTypeError("Must be at least 1")\n    return parsed\n\n\ndef nonnegative_int(value):\n    parsed = int(value)\n    if parsed < 0:\n        raise argparse.ArgumentTypeError("Must be nonnegative")\n    return parsed\n\n\ndef parse_args(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--endpoint", default="http://127.0.0.1:8001/v1")\n    parser.add_argument("--allow-remote", action="store_true",\n                        help="Permit a remote HTTPS endpoint in this standalone benchmark only")\n    parser.add_argument("--model", required=True, help="Exact served model ID from /v1/models")\n    parser.add_argument("--api-key-env", default="SOVEREIGN_VLLM_API_KEY",\n                        help="Environment variable containing the API key (never written to results)")\n    parser.add_argument("--repeats", type=positive_int, default=20)\n    parser.add_argument("--warmups", type=nonnegative_int, default=2)\n    parser.add_argument("--max-tokens", type=positive_int, default=128)\n    parser.add_argument("--timeout", type=positive_int, default=120)\n    parser.add_argument("--server-label", default="unspecified",\n                        help="Record GPU type, model revision, quantization and server flags")\n    parser.add_argument("--output", type=Path, default=Path("vllm-latency.json"))\n    args = parser.parse_args(argv)\n    try:\n        args.endpoint = normalize_endpoint(args.endpoint, args.allow_remote)\n    except ValueError as exc:\n        parser.error(str(exc))\n    return args\n\n\ndef run(args):\n    api_key = os.environ.get(args.api_key_env, "local")\n    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())\n    # Verify the requested model before starting any measured generation.\n    request = urllib.request.Request(args.endpoint + "/models",\n                                     headers={"Authorization": f"Bearer {api_key}"})\n    try:\n        with opener.open(request, timeout=args.timeout) as response:\n            models = json.load(response)["data"]\n        if args.model not in {item["id"] for item in models}:\n            raise ValueError("Requested model is not served by this endpoint")\n    except (OSError, ValueError, KeyError) as exc:\n        # Do not print HTTP response bodies or request headers containing credentials.\n        print(json.dumps({"preflight": "failed", "error_type": type(exc).__name__,\n                          "hint": "Start vLLM and check endpoint, served model ID and API key"}))\n        return 2\n    report = {\n        "schema_version": 1, "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "scope": "Client-observed streaming inference; includes network, server queueing and generation. Excludes application retrieval/tools, UI and server startup/model loading.",\n        "endpoint": args.endpoint, "model": args.model, "server_label": args.server_label,\n        "served_model_metadata": [{key: item[key] for key in ("id", "root", "max_model_len") if key in item}\n                                  for item in models if item["id"] == args.model],\n        "client_hardware": hardware_info(),\n        "settings": {"repeats": args.repeats, "warmups": args.warmups,\n                     "max_tokens": args.max_tokens, "temperature": 0, "concurrency": 1,\n                     "thinking": False, "prompt": PROMPT},\n        "measurement_notes": [\n            "TTFT ends at the first visible content chunk; reasoning chunks do not count.",\n            "Each request opens a connection; client network/connection overhead is included.",\n            "Decode speed uses (completion_tokens - 1)/(last_content - first_content); chunking makes it an estimate.",\n            "Missing usage remains unknown. Failed requests are excluded from latency percentiles.",\n            "Repeated identical inputs may hit server prefix caches; record cache settings in server-label.",\n            "Client GPU metadata describes the server only when the benchmark runs on that server.",\n            "Nonempty output is checked; arithmetic correctness and workflow quality are not evaluated.",\n            "Warmups are excluded from summary; these are warm serving samples, not cold-start measurements."],\n        "warmup_samples": [], "samples": [],\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    for index in range(args.warmups + args.repeats):\n        warmup = index < args.warmups\n        try:\n            sample = stream_sample(opener, args.endpoint, args.model, api_key,\n                                   args.max_tokens, args.timeout)\n        except (OSError, ValueError, KeyError, TypeError, IndexError) as exc:\n            sample = {"success": False, "error_type": type(exc).__name__}\n        sample["index"] = index + 1 if warmup else index - args.warmups + 1\n        report["warmup_samples" if warmup else "samples"].append(sample)\n        report["summary"] = summarize(report["samples"])\n        args.output.write_text(json.dumps(report, indent=2), encoding="utf-8")\n        print(json.dumps({"warmup": warmup, **sample}), flush=True)\n    print(json.dumps({"summary": report["summary"], "output": str(args.output)}), flush=True)\n    return 0 if report["summary"]["failed"] == 0 else 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(run(parse_args()))\n'
(work / 'benchmark_vllm_latency.py').write_text(client_source)

In [ ]:
result_path = work / "vllm-latency.json"
result = subprocess.run(
    [python, str(work / "benchmark_vllm_latency.py"), "--endpoint", endpoint,
     "--model", MODEL, "--repeats", str(REPEATS), "--warmups", str(WARMUPS),
     "--max-tokens", str(MAX_TOKENS), "--server-label", gpu + "; " + " ".join(command),
     "--output", str(result_path)])
if result_path.exists():
    report = json.loads(result_path.read_text())
    print(json.dumps(report["summary"], indent=2))
else:
    raise RuntimeError("Preflight failed; no measured results were produced")
if result.returncode:
    print("Some requests failed; inspect samples before comparing results.")

In [ ]:
# Download results plus environment/launch details for a reproducible report.
archive = work / "results"
archive.mkdir(exist_ok=True)
for name in ("vllm-latency.json", "requirements-resolved.txt", "server-command.json", "server.log"):
    if (work / name).exists():
        shutil.copy2(work / name, archive / name)
zip_path = shutil.make_archive(str(work / "vllm-latency-results"), "zip", archive)
from google.colab import files
files.download(zip_path)

In [ ]:
# Stop the process we started. Also disconnect/delete the Colab runtime.
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
if "log" in globals():
    log.close()
print("vLLM server stopped")